# 08.5 — RV Forecasting: Adding NB06/07 Surface-Quality Features to M4

**Question:** Do near-arbitrage stress indicators (`max_cond1`) and SSVI fit quality (`rmse_iv`)
add predictive power over the best SSVI-HAR model (M4)?

**M4_smooth** (best model from NB08):
`HAR3 + log_ATM + log_ATM×smooth_skew + log_η`  — R²_OOS = 0.868 at h=20

**New features tested:**
| Feature | Source | Economic meaning | Corr with RV20 |
|---------|--------|-----------------|----------------|
| `max_cond1` | NB06 | Near-butterfly-arbitrage stress — surface stretched | +0.234 |
| `log_rmse_iv` | NB06 | SSVI fit quality — higher = worse fit = noisier signal | +0.224 |
| `log_ATM × max_cond1` | computed | Joint: high surface level AND near-arbitrage state | — |

**Hypothesis:** When the surface is near an arbitrage violation (`max_cond1` high), the market
is in an extreme state that HAR lags may not fully capture → higher future RV.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys; sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None
import numpy as np, pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from scipy import stats as _scipy_stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("output")
PLOT_DIR   = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def _r2oos(y, yp, yn):
    return float(1-np.sum((y-yp)**2)/np.sum((y-yn)**2)) if np.sum((y-yn)**2)>0 else np.nan

def _dm(ea, eb, h):
    d = ea**2 - eb**2; T = len(d)
    if T < 4: return np.nan, np.nan
    vd = np.var(d, ddof=1)/T
    if vd <= 0: return np.nan, np.nan
    t_s = float(np.mean(d)/np.sqrt(vd)*np.sqrt((T+1-2*h+h*(h-1)/T)/T))
    return t_s, float(2*_scipy_stats.t.sf(abs(t_s), df=T-1))


In [ ]:
# ── Load and reconstruct df_sp (same as NB08 standalone) ────────────────────
df_ssvi = pd.read_csv(OUTPUT_DIR / "ssvi_forecasting_dataset.csv")
BASE_DATE = pd.Timestamp("2010-01-04")
df_ssvi["date"] = BASE_DATE + pd.to_timedelta(df_ssvi["time_elapsed"], unit="D")
df_ssvi["date"] = pd.to_datetime(df_ssvi["date"])

mkt = pd.read_csv(OUTPUT_DIR / "_cache_mh_fred.csv", index_col=0, parse_dates=True)
mkt.index = pd.to_datetime(mkt.index).normalize()
mkt["log_ret_sp"] = np.log(mkt["SP500"] / mkt["SP500"].shift(1))
mkt["sq_ret"] = mkt["log_ret_sp"]**2
for _h in [1,5,20]:
    mkt[f"rv_fwd{_h}"]  = np.sqrt(mkt["sq_ret"].rolling(_h).sum().shift(-_h)*(252/_h))
    mkt[f"lrv_fwd{_h}"] = np.log(mkt[f"rv_fwd{_h}"].clip(1e-10))
mkt = mkt.ffill()
mkt_r = mkt.reset_index().rename(columns={mkt.index.name or "index":"date"})
mkt_r["date"] = pd.to_datetime(mkt_r["date"])

df_mh = pd.merge_asof(df_ssvi.sort_values("date"), mkt_r.sort_values("date"),
                      on="date", direction="nearest", tolerance=pd.Timedelta("3D"))

# HAR components
df_mh["RV1"]        = np.sqrt(252) * df_mh["log_return"].abs()
df_mh["RV5_corsi"]  = df_mh["RV1"].rolling(5,  min_periods=5).mean()
df_mh["RV22_corsi"] = df_mh["RV1"].rolling(22, min_periods=22).mean()
df_mh["lrv1"]       = np.log(df_mh["RV1"].clip(1e-10))
df_mh["lrv5"]       = np.log(df_mh["RV5_corsi"].clip(1e-10))
df_mh["lrv22"]      = np.log(df_mh["RV22_corsi"].clip(1e-10))
df_mh["lrv_now"]    = df_mh["lrv1"]
_T30 = 30/365
df_mh["atm_ssvi"]  = np.exp(df_mh["alpha"]/2)*(_T30**(df_mh["beta"]/2-0.5))
df_mh = df_mh.dropna(subset=["lrv1","lrv5","lrv22","VIX","lrv_fwd1","lrv_fwd5","lrv_fwd20"]).reset_index(drop=True)

df_sp = df_mh.copy()
_trm  = df_sp["sample"]=="train"
_vixq75 = df_sp.loc[_trm,"VIX"].quantile(0.75)
df_sp["hv"]              = (df_sp["VIX"]>_vixq75).astype(float)
df_sp["log_ATM"]         = np.log(df_sp["atm_ssvi"].clip(1e-6))
df_sp["log_ATM_x_hv_atm"]= df_sp["log_ATM"] * (df_sp["atm_ssvi"]>df_sp.loc[_trm,"atm_ssvi"].quantile(0.75)).astype(float)
df_sp["log_eta"]         = np.log(df_sp["eta"].clip(1e-6))

# Smooth skew regime (M4 ingredient)
_rho_mean = df_sp.loc[_trm,"rho"].mean()
_rho_std  = df_sp.loc[_trm,"rho"].std()
df_sp["smooth_skew"]      = np.exp(-(df_sp["rho"]-_rho_mean)/_rho_std)
df_sp["log_ATM_x_smooth"] = df_sp["log_ATM"] * df_sp["smooth_skew"]

print(f"df_sp shape: {df_sp.shape}  train={_trm.sum()}  test={(~_trm).sum()}")


In [ ]:
# ── New features from NB06/07 ────────────────────────────────────────────────
# max_cond1: butterfly near-arbitrage (higher = surface more stretched/stressed)
# rmse_iv:   SSVI fit quality (higher = worse fit, noisier surface signal)

df_sp["log_max_cond1"]    = np.log(df_sp["max_cond1"].clip(1e-6))
df_sp["log_rmse_iv"]      = np.log(df_sp["rmse_iv"].clip(1e-6))
df_sp["log_ATM_x_cond1"]  = df_sp["log_ATM"] * df_sp["max_cond1"]

# Correlation check on train
tr = df_sp[df_sp["sample"]=="train"]
print("Feature correlations with lrv_fwd5 (train):")
for f in ["log_max_cond1","log_rmse_iv","log_ATM_x_cond1","max_cond1","rmse_iv","skew_stress"]:
    if f in tr.columns:
        print(f"  {f:28s}  r={tr[f].corr(tr['lrv_fwd5']):+.4f}")


In [ ]:
# ── Model evaluation ─────────────────────────────────────────────────────────
HAR3  = ["lrv1","lrv5","lrv22"]
M4_feats = HAR3 + ["log_ATM","log_ATM_x_smooth","log_eta"]

SPECS = {
    "M4_smooth       [best]":    M4_feats,
    "M4+cond1_lin              ": M4_feats + ["log_max_cond1"],
    "M4+rmse_lin               ": M4_feats + ["log_rmse_iv"],
    "M4+cond1+rmse             ": M4_feats + ["log_max_cond1","log_rmse_iv"],
    "M4+cond1_interact         ": M4_feats + ["log_ATM_x_cond1"],
    "M4+skew_stress            ": M4_feats + ["skew_stress"],
    "M4+all_new                ": M4_feats + ["log_max_cond1","log_rmse_iv","skew_stress"],
}
HORIZONS = [(1,"lrv_fwd1"),(5,"lrv_fwd5"),(20,"lrv_fwd20")]
_BM = "M4_smooth       [best]"

rows=[]; preds={}
for h, tgt in HORIZONS:
    _tr = df_sp[df_sp["sample"]=="train"].dropna(subset=[tgt]).reset_index(drop=True)
    _te = df_sp[df_sp["sample"]=="test"].dropna(subset=[tgt]).reset_index(drop=True)
    y_tr=_tr[tgt].values; y_te=_te[tgt].values; naive=_te["lrv_now"].values
    for sn,feats in SPECS.items():
        avail=[f for f in feats if f in _tr.columns]
        m=LinearRegression().fit(_tr[avail].fillna(0), y_tr)
        p=m.predict(_te[avail].fillna(0))
        preds[(sn,h)]=p
        n=min(len(y_te),len(p))
        rows.append(dict(h=h,spec=sn,nf=len(avail),
                         R2=_r2oos(y_te[:n],p[:n],naive[:n]),
                         RMSE=float(np.sqrt(np.mean((y_te[:n]-p[:n])**2)))))

# DM vs M4
for h,tgt in HORIZONS:
    _te=df_sp[df_sp["sample"]=="test"].dropna(subset=[tgt]).reset_index(drop=True)
    y_te=_te[tgt].values; pm4=preds.get((_BM,h))
    for sn in SPECS:
        pn=preds.get((sn,h))
        if pn is None or pm4 is None: continue
        n=min(len(y_te),len(pn),len(pm4))
        ds,dp=_dm(y_te[:n]-pn[:n], y_te[:n]-pm4[:n], h=h)
        for r in rows:
            if r["h"]==h and r["spec"]==sn:
                r["DM"]=ds; r["DMp"]=dp; break

res=pd.DataFrame(rows)

print("="*72)
print("NB06/07 FEATURES ADDED TO M4 — R2_OOS  (DM vs M4: neg=better)")
for h in [1,5,20]:
    sub=res[res["h"]==h]; m4r=sub.loc[sub["spec"]==_BM,"R2"].values[0]
    print(f"
--- h={h}  M4 R2={m4r:.4f} ---")
    print(f"  {'Spec':36s}  nf  {'R2':>8}  {'Delta':>7}  {'DM':>7}  p")
    for _,r in sub.sort_values("R2",ascending=False).iterrows():
        d=r["R2"]-m4r
        dm_s=f"{r['DM']:+.3f}" if not np.isnan(r.get("DM",float("nan"))) else "   ref"
        ps=f"{r['DMp']:.4f}" if not np.isnan(r.get("DMp",float("nan"))) else "     -"
        sig=("***" if r.get("DMp",1)<0.01 else("**" if r.get("DMp",1)<0.05 else("*" if r.get("DMp",1)<0.1 else"")))
        print(f"  {r['spec']:36s}  {r['nf']:2d}  {r['R2']:>8.4f}  {d:>+7.4f}  {dm_s:>7}  {ps}{sig}")


In [ ]:
# ── OLS coefficients for best additions (h=5, HAC NW) ──────────────────────
print("="*65)
print("OLS coefficients (h=5, HAC Newey-West) — new features")
_tr5=df_sp[df_sp["sample"]=="train"].dropna(subset=["lrv_fwd5"]).reset_index(drop=True)
for sn,feats in [
    ("M4_smooth [best]", M4_feats),
    ("M4+cond1_lin",     M4_feats+["log_max_cond1"]),
    ("M4+rmse_lin",      M4_feats+["log_rmse_iv"]),
    ("M4+cond1+rmse",    M4_feats+["log_max_cond1","log_rmse_iv"]),
]:
    avail=[f for f in feats if f in _tr5.columns]
    Xc=sm.add_constant(_tr5[avail].fillna(0).values)
    m=sm.OLS(_tr5["lrv_fwd5"].values,Xc).fit(cov_type="HAC",cov_kwds={"maxlags":5})
    print(f"
  {sn}  R2_IS={m.rsquared:.4f}")
    for f,c,t,p in zip(["const"]+avail,m.params,m.tvalues,m.pvalues):
        sig="***" if p<0.01 else("**" if p<0.05 else("*" if p<0.1 else""))
        print(f"    {f:30s}  {c:+.4f}  (t={t:+.2f}{sig})")


In [ ]:
# ── Summary bar chart ────────────────────────────────────────────────────────
_models_plot = [
    "M4_smooth       [best]",
    "M4+cond1_lin              ",
    "M4+rmse_lin               ",
    "M4+cond1+rmse             ",
    "M4+cond1_interact         ",
    "M4+all_new                ",
]
_colors = ["#e08030","#4477aa","#229955","#cc3311","#884499","#888888"]
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,(h,_) in zip(axes,HORIZONS):
    sub=res[(res["h"]==h)&(res["spec"].isin(_models_plot))].set_index("spec")
    sub=sub.reindex([m for m in _models_plot if m in sub.index])
    xp=np.arange(len(sub))
    m4r=sub.loc[_BM,"R2"] if _BM in sub.index else 0
    bars=ax.bar(xp,sub["R2"].values,color=_colors[:len(sub)],width=0.65,edgecolor="white",alpha=0.85)
    ax.axhline(m4r,color="#e08030",lw=1.2,ls="--",alpha=0.7)
    for bar,idx in zip(bars,sub.index):
        d=sub.loc[idx,"R2"]-m4r
        if abs(d)>0.0001:
            ax.text(bar.get_x()+bar.get_width()/2, sub.loc[idx,"R2"]+0.001,
                    f"{d:+.4f}",ha="center",va="bottom",fontsize=7.5,
                    color="#229955" if d>=0 else "#cc3311")
    ax.set_xticks(xp)
    ax.set_xticklabels([s.strip() for s in sub.index],rotation=38,ha="right",fontsize=7.5)
    ax.set_title(f"h={h}",fontsize=10); ax.set_ylabel("R2_OOS")
    ax.grid(True,axis="y",alpha=0.25)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.suptitle("NB06/07 features added to M4_smooth  (DM dashed line = M4 baseline)",
             fontsize=11,fontweight="bold",y=1.02)
plt.tight_layout()
fpath=PLOT_DIR/"rv_features_nb0607.png"
plt.savefig(fpath,dpi=130,bbox_inches="tight"); plt.show()
print(f"Saved: {fpath}")
